## Cell 1 — Kiểm tra GPU & Cài đặt

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError("❌ Chưa bật GPU! Vào Runtime → Change runtime type → T4 GPU")
print(result.stdout.split('\n')[0])
print("✅ GPU OK")

!pip install -q -U transformers datasets accelerate bitsandbytes peft librosa evaluate jiwer

## Cell 2 — Tự động phát hiện Part & Cấu hình

In [ ]:
import os
import glob
import torch
from google.colab import drive

drive.mount('/content/drive')

# ─── THÔNG SỐ CỐ ĐỊNH ─────────────────────────────────────────
MODEL_ID       = "openai/whisper-large-v3"
BASE_DIR       = "/content/drive/MyDrive/ASR_Continual_Learning"
TRAIN_SIZE     = 2000
VAL_SIZE       = 200
BATCH_SIZE     = 8
GRAD_ACCUM     = 2            # effective batch = 16
LEARNING_RATE  = 1e-4
TOTAL_STEPS    = TRAIN_SIZE // (BATCH_SIZE * GRAD_ACCUM)  # = 125
WARMUP_STEPS   = max(10, TOTAL_STEPS // 10)

# ─── TỰ ĐỘNG PHÁT HIỆN PART ───────────────────────────────────
existing = sorted(glob.glob(os.path.join(BASE_DIR, "checkpoint_part_*")))

if len(existing) == 0:
    # Chưa có part nào → bắt đầu từ part 1
    CURRENT_PART   = 1
    PREV_CKPT_PATH = None
else:
    # Lấy part lớn nhất đã có → train part tiếp theo
    last_part      = max([int(p.split("_")[-1]) for p in existing])
    CURRENT_PART   = last_part + 1
    PREV_CKPT_PATH = os.path.join(BASE_DIR, f"checkpoint_part_{last_part}")

OUTPUT_DIR = os.path.join(BASE_DIR, f"checkpoint_part_{CURRENT_PART}")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Data offset: mỗi part lấy 2000 mẫu tiếp theo
SKIP_TRAIN = (CURRENT_PART - 1) * TRAIN_SIZE
SKIP_VAL   = (CURRENT_PART - 1) * VAL_SIZE

# In tóm tắt
print("=" * 55)
print(f"  CONTINUAL LEARNING — PART {CURRENT_PART}")
print("=" * 55)
print(f"  Load từ     : {PREV_CKPT_PATH or 'Base model (OpenAI Hub)'}")
print(f"  Lưu ra      : {OUTPUT_DIR}")
print(f"  Train mẫu   : skip {SKIP_TRAIN} → lấy {TRAIN_SIZE} mẫu")
print(f"  Val mẫu     : skip {SKIP_VAL} → lấy {VAL_SIZE} mẫu")
print(f"  Total steps : {TOTAL_STEPS}")
print("=" * 55)

## Cell 3 — Load Model & Gắn LoRA

In [ ]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model, PeftModel

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

if PREV_CKPT_PATH is None:
    # ── PART 1: Load processor từ Hub, khởi tạo LoRA mới ──
    print(f"[Part 1] Khởi tạo LoRA mới từ base model...")
    processor = WhisperProcessor.from_pretrained(MODEL_ID, language="en", task="transcribe")

    base_model = WhisperForConditionalGeneration.from_pretrained(
        MODEL_ID, quantization_config=bnb_config, device_map="auto"
    )
    base_model = prepare_model_for_kbit_training(base_model)

    lora_config = LoraConfig(
        r=32, lora_alpha=64,
        target_modules=["q_proj", "v_proj"],
        lora_dropout=0.05, bias="none"
    )
    model = get_peft_model(base_model, lora_config)

else:
    # ── PART 2+: Load adapter từ part trước, train tiếp ──
    print(f"[Part {CURRENT_PART}] Load adapter từ: {PREV_CKPT_PATH}")
    processor = WhisperProcessor.from_pretrained(PREV_CKPT_PATH)

    base_model = WhisperForConditionalGeneration.from_pretrained(
        MODEL_ID, quantization_config=bnb_config, device_map="auto"
    )
    base_model = prepare_model_for_kbit_training(base_model)

    # is_trainable=True: tiếp tục train adapter cũ
    model = PeftModel.from_pretrained(base_model, PREV_CKPT_PATH, is_trainable=True)

model.config.use_cache = False
model.print_trainable_parameters()
print("✅ Model sẵn sàng!")

## Cell 4 — Chuẩn bị Dataset

In [ ]:
from dataclasses import dataclass
from typing import Any, Dict, List
from datasets import load_dataset

def prepare_dataset(batch):
    audio = batch["audio"]
    batch["input_features"] = processor(
        audio["array"], sampling_rate=audio["sampling_rate"]
    ).input_features[0]
    batch["labels"] = processor.tokenizer(batch["text"]).input_ids
    return batch

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, Any]:
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")
        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all():
            labels = labels[:, 1:]
        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

# Load đúng đoạn dữ liệu của part này
print(f"Đang nạp dữ liệu Part {CURRENT_PART}...")
print(f"  Train: bỏ {SKIP_TRAIN} → lấy {TRAIN_SIZE} mẫu")
print(f"  Val  : bỏ {SKIP_VAL} → lấy {VAL_SIZE} mẫu")

raw_train = load_dataset("openslr/librispeech_asr", "clean", split="train.100", streaming=True)
raw_val   = load_dataset("openslr/librispeech_asr", "clean", split="validation",  streaming=True)

train_dataset = raw_train.skip(SKIP_TRAIN).take(TRAIN_SIZE).map(
    prepare_dataset, remove_columns=list(raw_train.features.keys())
)
val_dataset = raw_val.skip(SKIP_VAL).take(VAL_SIZE).map(
    prepare_dataset, remove_columns=list(raw_val.features.keys())
)

print("✅ Dataset sẵn sàng!")

## Cell 5 — Train & Lưu Adapter

In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, GenerationConfig

print("=" * 55)
print(f"  BẮT ĐẦU TRAIN PART {CURRENT_PART} — {TOTAL_STEPS} steps")
print("=" * 55)

training_args = Seq2SeqTrainingArguments(
    output_dir="/content/train_tmp",
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    max_steps=TOTAL_STEPS,
    warmup_steps=WARMUP_STEPS,
    eval_strategy="steps",
    eval_steps=TOTAL_STEPS,
    save_strategy="no",
    predict_with_generate=True,
    fp16=True,
    logging_steps=25,
    remove_unused_columns=False,
    label_names=["labels"],
    report_to="none",
)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    processing_class=processor.feature_extractor,
)

trainer.train()

# ─── Save đúng cách ───────────────────────────────────────────
print(f"\nĐang lưu Part {CURRENT_PART} ra Drive...")

# 1. LoRA adapter weights
model.save_pretrained(OUTPUT_DIR)

# 2. Processor (tokenizer + feature extractor)
processor.save_pretrained(OUTPUT_DIR)

# 3. Generation config với forced_decoder_ids
forced_decoder_ids = processor.get_decoder_prompt_ids(language="en", task="transcribe")
gen_config = GenerationConfig.from_pretrained(MODEL_ID)
gen_config.forced_decoder_ids = forced_decoder_ids
gen_config.suppress_tokens = []
gen_config.save_pretrained(OUTPUT_DIR)

# In danh sách file đã lưu
print("\n" + "=" * 55)
print(f"✅ PART {CURRENT_PART} HOÀN TẤT! Đã lưu tại:")
print(f"   {OUTPUT_DIR}")
print("\nCác file đã lưu:")
for f in sorted(os.listdir(OUTPUT_DIR)):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f))
    print(f"   {f:45s} {size/1024/1024:.1f} MB")
print("=" * 55)
print(f"\n👉 Lần chạy tiếp theo sẽ tự động train PART {CURRENT_PART + 1}")
print(f"   với mẫu train [{SKIP_TRAIN + TRAIN_SIZE} → {SKIP_TRAIN + TRAIN_SIZE * 2})")